# Double-checking citations

**Difficulty:** Beginner

Verify a claim against its source before trusting it — the pattern this whole project has secretly been running since lesson 1, when we caught the vendor's own "0% hallucination" marketing claim failing to hold up against the technical docs. This cookbook automates exactly that check.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/citation_check (click to expand)</summary>

# Double-checking citations

> Catch wrong or hallucinated citations by checking against the source document. One TypeSafe Choice question decides whether the quote's context supports the claim, and its confidence can flag the citation for human review.

An LLM answers a question and attaches citations: for each claim, a section of a source
document and the quote it rests on. Some of those citations are wrong or hallucinated:
the quote can be missing from the document altogether, or sit in it word for word while
its context says the opposite of the claim.

Checking one by hand is slow: find the document, find the quote inside it, then read
enough of its context to tell whether it backs the claim up.

To automate that check, we first look for missing quotes with an ordinary string match,
and then we use a `Choice` question to read each surviving quote's context and decide
whether it supports the claim.

`check_citation()`, the function this cookbook builds, takes a source document and one
citation and returns one of four verdicts: `verified`, `unsupported`, `contradicted`, or
`fabricated`. It also returns a confidence that flags the ones a human should look at.

## Find each quote in the source

A quote that is not in the source is fabricated, and no model is needed to find that out.
Normalize whitespace and curly quotes so a quote still matches across line wraps, then
look for it as a substring.

A citation can name a section without quoting anything from it. There is nothing to match
in that case, so take the section the citation names and go straight to the model.

## Verify whether the source supports the claim

A citation that still has a quote at this point matches the source word for word. That is
**not enough** — the quote can be accurate and the claim built on top of it still wrong.
Deciding that takes the quote's context.

One `Choice` question per surviving citation covers the three ways a section can relate
to a claim — `supports`, `contradicts`, or `says_nothing` — which map onto three of the
four final verdicts (`fabricated` is decided by the string match alone, before any model
call happens):

```
relation: Choice(
    instructions="How does the section relate to the claim?",
    criteria={
        "supports": "The section states the claim or directly implies that it is true",
        "contradicts": "The section states the opposite of the claim or implies it is false",
        "says_nothing": "The section does not address what the claim asserts, either way",
    },
)
```

`AUTO_ACCEPT` (0.8 in the cookbook) decides what happens to each verdict: at or above that
confidence, the verdict stands on its own; below it, a human confirms the verdict before
anything acts on it. Start high, and lower the threshold as you see how the model does on
your own documents.

## Worked example: eight citations against RFC 7519

The cookbook runs eight real citations against RFC 7519 (JSON Web Token) — four accurate,
four deliberately broken — through this exact check:

```
citation          quote         relation        conf  verdict       action
epoch_seconds     found         supports        0.93  verified        auto
aud_reject        found         supports        0.95  verified        auto
sig_reporting     missing       -                  -  fabricated      auto
clock_skew        found         supports        0.99  verified        auto
exp_required      found         contradicts     0.99  contradicted    auto
pii_encryption    found         says_nothing    0.27  unsupported   review
iat_future        section-only  says_nothing    0.56  unsupported   review
duplicate_names   found         supports        0.99  verified        auto
```

`sig_reporting` never reached the model — its quote isn't in the RFC at all, so the string
match alone marks it `fabricated`. `exp_required` quotes its section word for word, but
that same section says the opposite of the claim, so it's `contradicted` at confidence
0.99 — the case that proves string-matching alone isn't enough. `pii_encryption` and
`iat_future` both came back `unsupported`, at confidence low enough (0.27 and 0.56) to
fall under the threshold and go to a human instead of standing automatically.

The string match is exact after normalization: a quote that's truncated or lightly
reworded comes back `fabricated` too. A production system that tolerates sloppy quoting
would need fuzzy matching instead.

> The full cookbook wires this against a real RFC document with its own parsing code, a
> disk-backed cache, and a playground link — trimmed here to the check itself, since that
> file-loading infrastructure isn't something this repo carries. Read the full cookbook at
> [docs.typesafe.ai/cookbooks/citation_check](https://docs.typesafe.ai/cookbooks/citation_check) any time.

</details>

## The core idea

Two steps, in a specific order, for a reason:

1. **String match first, no model call.** A quote that literally isn't in the source is fabricated — that's a fact you can check with `.includes()`, not a judgment. Lesson 1's design guidance ("code owns the workflow, System One handles narrow, structured decisions") applies directly: don't spend a model call on something ordinary code already knows for certain.
2. **`Choice` only for what's left.** Once a quote is confirmed present, the real question is whether its *context* actually backs the claim — that's genuinely ambiguous, and that's what the model is for. One `Choice` between `supports` / `contradicts` / `says_nothing`, mapped onto the verdicts `verified` / `contradicted` / `unsupported`.

Then confidence-gated routing (`lessons/patterns/0007-confidence-gated-routing.ipynb`) decides what happens next: high-confidence verdicts stand on their own, low-confidence ones go to a human. `fabricated` never even reaches that gate — it was decided in step 1, with no confidence to report at all.

In [1]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const source = `
Refunds are issued within 5 business days of approval. Refund requests must be
submitted within 30 days of the original purchase. Store credit does not expire.
Digital purchases are non-refundable once downloaded.
`;

const citations = [
  {
    id: "refund_window",
    claim: "Customers have a month to request a refund.",
    quote: "Refund requests must be submitted within 30 days of the original purchase.",
  },
  {
    id: "digital_refundable",
    claim: "Digital purchases can always be refunded.",
    quote: "Digital purchases are non-refundable once downloaded.",
  },
  {
    id: "credit_expires",
    claim: "Store credit expires after a year.",
    quote: "Store credit is only valid for 12 months from issue.",
  },
  {
    id: "unrelated_claim",
    claim: "Refunds are processed by a third-party payment provider.",
    quote: "Refunds are issued within 5 business days of approval.",
  },
];

const AUTO_ACCEPT = 0.8;

function normalize(text: string): string {
  return text.replace(/\s+/g, " ").trim();
}

async function checkCitation(citation: typeof citations[number]) {
  // Step 1: cheap string match, no model call - catches fabricated quotes for free
  if (!normalize(source).includes(normalize(citation.quote))) {
    return { id: citation.id, verdict: "fabricated", confidence: null, action: "auto" };
  }

  // Step 2: the quote is genuinely there - but does its context support the claim?
  const response = await client.systemOne({
    state: { claim: citation.claim, source },
    questions: {
      relation: choice("How does the source relate to the claim?", {
        supports: "The source states the claim or directly implies that it is true",
        contradicts: "The source states the opposite of the claim or implies it is false",
        saysNothing: "The source does not address what the claim asserts, either way",
      }),
    },
  });

  const { relation } = response.answers;
  const verdictFor = {
    supports: "verified",
    contradicts: "contradicted",
    saysNothing: "unsupported",
  } as const;

  return {
    id: citation.id,
    verdict: verdictFor[relation.choice],
    confidence: relation.confidence,
    action: relation.confidence >= AUTO_ACCEPT ? "auto" : "review",
  };
}

for (const citation of citations) {
  console.log(await checkCitation(citation));
}

{
  id: "refund_window",
  verdict: "verified",
  confidence: 1,
  action: "auto"
}
{
  id: "digital_refundable",
  verdict: "contradicted",
  confidence: 1,
  action: "auto"
}
{
  id: "credit_expires",
  verdict: "fabricated",
  confidence: null,
  action: "auto"
}
{
  id: "unrelated_claim",
  verdict: "unsupported",
  confidence: 1,
  action: "auto"
}


## What just happened

Four citations against a made-up refund policy, hitting all four verdicts on purpose:

- `refund_window` — quote's there, and it genuinely backs the claim → `verified`
- `digital_refundable` — quote's there, but it says the *opposite* of the claim → `contradicted` (the case that proves string-matching alone isn't enough)
- `credit_expires` — that exact sentence never appears in the source at all → `fabricated`, caught before any model call
- `unrelated_claim` — the quote is real, but it's silent on *who* processes refunds → `unsupported`

**Try it yourself:** predict `unrelated_claim`'s confidence before running it — a "says nothing either way" verdict is inherently murkier than a clean support or contradiction, so expect it lower than the other two model-judged cases. Then write a fifth citation of your own, deliberately near the `AUTO_ACCEPT` boundary, and see which side of `0.8` it lands on.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the auto/review split this cookbook's `AUTO_ACCEPT` threshold is built on
- `research/typesafe-ai.md` — the actual marketing-claim-vs-technical-docs finding this cookbook automates a version of

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.